In [1]:
# SHAZAM SHM MODELS 
# Este script aplica tres enfoques complementarios para analizar las mutaciones somáticas 
# en tu repertorio: primero calcula la carga mutacional global comparando cada secuencia 
# completa contra su germline, luego evalúa la distribución de mutaciones por 
# subregiones (CDR y FWR según definiciones IMGT) para distinguir entre regiones de unión
# y estructurales, y finalmente clasifica las mutaciones según propiedades fisicoquímicas
# de los aminoácidos (carga, polaridad, hidrofobicidad o volumen) para estimar su impacto
# funcional

In [2]:

library(shazam)
library(alakazam)
library(readr)
library(dplyr)
library(ggplot2)


Loading required package: ggplot2

Warning message:
"replacing previous import 'S4Arrays::makeNindexFromArrayViewport' by 'DelayedArray::makeNindexFromArrayViewport' when loading 'SummarizedExperiment'"


To cite the SHazaM package in publications, please use:

  Gupta N, Vander Heiden J, Uduman M, Gadala-Maria D, Yaari G,
  Kleinstein S (2015). "Change-O: a toolkit for analyzing large-scale B
  cell immunoglobulin repertoire sequencing data." _Bioinformatics_,
  1-3. doi:10.1093/bioinformatics/btv359
  <https://doi.org/10.1093/bioinformatics/btv359>.

To cite the selection analysis methods, please use:

  Yaari G, Uduman M, Kleinstein S (2012). "Quantifying selection in
  high-throughput Immunoglobulin sequencing data sets." _Nucleic acids
  research_, *40*(17), e134. doi:10.1093/nar/gks457
  <https://doi.org/10.1093/nar/gks457>.

To cite the HH_S5F model and the targeting model generation methods,
please use:

  Yaari G, Vander Heiden J, Uduman M, Gadala-Maria D, Gupta N, Stern J,
  

In [3]:
# MUTATION ANALYSIS SHAZAM
# 1. Cargar tu archivo
archivo_clones <- "../data/output/repertorio_C_insilico_100_seqs_clone-pass.tsv"
clones <- read_tsv(archivo_clones) %>%
  mutate(sample_id = "repertorio_simulado")

# 2. Mutaciones globales (toda la secuencia)
mut_global <- observedMutations(
  clones,
  sequenceColumn="sequence_alignment",
  germlineColumn="germline_alignment",
  regionDefinition=NULL,   # toda la secuencia
  frequency=TRUE
)

# 3. Mutaciones por subregión (CDR/FWR según IMGT)
mut_regions <- observedMutations(
  clones,
  sequenceColumn="sequence_alignment",
  germlineColumn="germline_alignment",
  regionDefinition=IMGT_V_BY_REGIONS,   # esquema IMGT con CDR/FWR
  frequency=TRUE
)


# Mutaciones globales
db_all <- mut_global[, c("mu_freq_seq_r","mu_freq_seq_s")]

# Mutaciones en CDR1 y CDR2 (R y S)
cdr_mutations <- mut_regions[, c("mu_freq_cdr1_r","mu_freq_cdr1_s",
                                 "mu_freq_cdr2_r","mu_freq_cdr2_s")]

# Mutaciones en FWR1–3 (R y S)
fwr_mutations <- mut_regions[, c("mu_freq_fwr1_r","mu_freq_fwr1_s",
                                 "mu_freq_fwr2_r","mu_freq_fwr2_s",
                                 "mu_freq_fwr3_r","mu_freq_fwr3_s")]

# Combinar todo en un solo objeto
db_selected <- cbind(db_all, cdr_mutations, fwr_mutations)

# Revisar primeras filas
head(db_selected)



Rows: 100 Columns: 50
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (20): sequence_id, sequence, v_call, d_call, j_call, sequence_alignment,...
dbl (25): junction_length, np1_length, np2_length, v_sequence_start, v_seque...
lgl  (5): rev_comp, productive, stop_codon, vj_in_frame, c_call

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


,mu_freq_seq_r,mu_freq_seq_s,mu_freq_cdr1_r,mu_freq_cdr1_s,mu_freq_cdr2_r,mu_freq_cdr2_s,mu_freq_fwr1_r,mu_freq_fwr1_s,mu_freq_fwr2_r,mu_freq_fwr2_s,mu_freq_fwr3_r,mu_freq_fwr3_s
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.013927577,0.002785515,0.12500000,0,0.00000000,0.00000000,0,0,0.01960784,0.00000000,0.00877193,0.00877193
2,0.013812155,0.000000000,0.00000000,0,0.00000000,0.00000000,0,0,0.01960784,0.00000000,0.01754386,0.00000000
3,0.005633803,0.002816901,0.00000000,0,0.00000000,0.04166667,0,0,0.01960784,0.00000000,0.00877193,0.00000000
4,0.013850416,0.008310249,0.00000000,0,0.04166667,0.00000000,0,0,0.00000000,0.01960784,0.01754386,0.00000000
5,0.011111111,0.005555556,0.00000000,0,0.00000000,0.00000000,0,0,0.00000000,0.00000000,0.01754386,0.00877193
6,0.005555556,0.002777778,0.03333333,0,0.00000000,0.04761905,0,0,0.00000000,0.00000000,0.00877193,0.00000000


In [7]:
readr::write_tsv(db_selected, "../results/shm_models/mutation_table/mutation_model_E_100seqs.tsv")

In [5]:
ruta <- "../data/output/"

depths <- c(
100,200,400,800,
1600,3200,6400,
12800,25600,51200,
102400
)


mutation_results <- list()


for(n in depths){

cat("Procesando E", n, "\n")


archivo <- paste0(
ruta,
"repertorio_E_insilico_",
n,
"_seqs_clone-pass.tsv"
)


if(!file.exists(archivo)){
next
}


# Cargar datos

db <- read_tsv(
archivo,
show_col_types = FALSE
)


db <- db %>%
mutate(
clone_id = as.character(clone_id)
) %>%
filter(!is.na(clone_id))


# Filtrar germline

db_shazam <- db %>%
filter(
!is.na(germline_alignment),
germline_alignment != ""
)


# Colapsar clones

clones <- collapseClones(
db_shazam,
cloneColumn="clone_id",
sequenceColumn="sequence_alignment",
germlineColumn="germline_alignment",
regionDefinition=IMGT_V,
method="thresholdedFreq",
minimumFrequency=0.6,
nproc=1
)


# Mutaciones globales

mut_global <- observedMutations(
clones,
sequenceColumn="sequence_alignment",
germlineColumn="germline_alignment",
regionDefinition=IMGT_V,
frequency=TRUE
)



# Mutaciones por regiones

mut_regions <- observedMutations(
clones,
sequenceColumn="sequence_alignment",
germlineColumn="germline_alignment",
regionDefinition=IMGT_V_BY_REGIONS,
frequency=TRUE
)



# Seleccionar variables

db_selected <- cbind(

mut_global[,c(
"mu_freq_seq_r",
"mu_freq_seq_s"
)],


mut_regions[,c(
"mu_freq_cdr1_r",
"mu_freq_cdr1_s",
"mu_freq_cdr2_r",
"mu_freq_cdr2_s"
)],


mut_regions[,c(
"mu_freq_fwr1_r",
"mu_freq_fwr1_s",
"mu_freq_fwr2_r",
"mu_freq_fwr2_s",
"mu_freq_fwr3_r",
"mu_freq_fwr3_s"
)]

)


# Guardar resultado en lista

nombre <- paste0(
"E_",
n
)


mutation_results[[nombre]] <- db_selected


}

Procesando E 100 


ERROR: [1m[33mError[39m in `mut_global[, c("mu_freq_seq_r", "mu_freq_seq_s")]`:[22m
[33m![39m Can't subset columns that don't exist.
[31mx[39m Columns `mu_freq_seq_r` and `mu_freq_seq_s` don't exist.


In [ ]:
mutation_results[[nombre]] <- db_selected
saveRDS(
  mutation_results,
  "../results/shm_models/mutation_regions_all_A.rds"
)
